# CKAN feature demo

This notebook uses HDX, a real public CKAN catalog. Change `CKAN_ORIGIN` to use another CKAN deployment. The first cells run public reads; the operation list shows every feature implemented by the package.

In [ ]:
# Optional, for standalone users only; skip this cell to reuse the current environment.
# !uv pip install datasluice
from collections import defaultdict
from collections.abc import Mapping

from datasluice import DataSluice, DirectResourceLocator
from datasluice.connectors.catalog.ckan import CKANClientSettings, create_sync_client
from datasluice.connectors.catalog.ckan.inventory import CKAN_ACTIONS
from datasluice.domain.catalog.auth import CKANCredential
from datasluice.domain.catalog.models import MappingRecord, NativeRecord, ValueRecord

CKAN_ORIGIN = "https://data.humdata.org"
CKAN_TOKEN = None
OUTPUT_DIR = "./data"
credential = CKANCredential(CKAN_TOKEN) if CKAN_TOKEN else None

In [ ]:
operations_by_group = defaultdict(list)
for action in CKAN_ACTIONS.entries:
    operations_by_group[action.group].append(action)

for group, actions in sorted(operations_by_group.items()):
    print(f"{group} ({len(actions)} operations)")
    for action in actions:
        print(f"  {action.name} [{action.mutation_class}]")

## Public reads

These calls exercise the discovery, dataset, resource, group, organization, tag, and license service families against a real CKAN catalog.

In [ ]:
with create_sync_client(CKANClientSettings(base_url=CKAN_ORIGIN, credential=credential)) as client:
    status = client.action_discovery.status_show()
    status_item = status.items[0]
    if not isinstance(status_item, MappingRecord):
        raise TypeError("status_show did not return a mapping record")
    datasets = client.datasets.package_search(q="climate", rows=1)
    dataset_search_item = datasets.items[0]
    if not isinstance(dataset_search_item, NativeRecord):
        raise TypeError("package_search did not return a native record")
    dataset_name = dataset_search_item.payload.get("name")
    if not isinstance(dataset_name, str):
        raise TypeError("package_search returned a dataset without a name")
    dataset = client.datasets.package_show(id=dataset_name)
    dataset_item = dataset.items[0]
    if not isinstance(dataset_item, NativeRecord):
        raise TypeError("package_show did not return a native record")
    resources = dataset_item.payload.get("resources")
    if not isinstance(resources, list | tuple) or not resources or not isinstance(resources[0], Mapping):
        raise TypeError("package_show returned a dataset without resources")
    resource_id = resources[0].get("id")
    if not isinstance(resource_id, str):
        raise TypeError("package_show returned a resource without an id")
    resource = client.resources.resource_show(id=resource_id)
    groups = client.groups.group_list(limit=3)
    organizations = client.organizations.organization_list(limit=3)
    tags = client.vocabularies_licenses.tag_list()
    licenses = client.vocabularies_licenses.license_list()

print(status_item.payload)
dataset_title = dataset_item.payload.get("title")
if not isinstance(dataset_title, str):
    raise TypeError("package_show returned a dataset without a title")
print(dataset_title)
resource_item = resource.items[0]
if not isinstance(resource_item, NativeRecord):
    raise TypeError("resource_show did not return a native record")
resource_url = resource_item.payload.get("url")
if not isinstance(resource_url, str):
    raise TypeError("resource_show returned a resource without a URL")
print(resource_url)
print(groups.items)
print(organizations.items)
print(len(tags.items), len(licenses.items))

## List every dataset on the portal

In [ ]:
with create_sync_client(CKANClientSettings(base_url=CKAN_ORIGIN, credential=credential)) as client:
    dataset_names = client.datasets.package_list()

for dataset in dataset_names.items:
    if not isinstance(dataset, ValueRecord):
        raise TypeError("package_list did not return scalar values")
    print(dataset.value)

print(f"{len(dataset_names.items)} datasets")

## Convert a CKAN resource

This is the direct-resource workflow: use any public CKAN resource URL and materialize it as Parquet.

In [ ]:
resource_item = resource.items[0]
if not isinstance(resource_item, NativeRecord):
    raise TypeError("resource_show did not return a native record")
RESOURCE_URL = resource_item.payload.get("url")
if not isinstance(RESOURCE_URL, str):
    raise TypeError("resource_show returned a resource without a URL")

with DataSluice() as ds:
    artifact = ds.materialize(DirectResourceLocator(uri=RESOURCE_URL), OUTPUT_DIR, mode="parquet")

print(artifact.uri)
print(artifact.content_digest)

## Authenticated, admin, and mutation operations

Set `CKAN_TOKEN` in the setup cell and point `CKAN_ORIGIN` at a controlled CKAN deployment to use the 74 standard and 6 destructive operations listed above. The public HDX catalog is read-only for this demo.